
# 練習問題: コマンドライン引数の処理

## 目標

コマンドラインから渡した引数 (文字列) を数値に変換して使う方法を身につける.

## 課題

`args.cpp` (または `args.f90`) は, 第1引数を整数 `n`, 第2引数を実数 `x` として受け取り, `x` の `n` 乗を表示する (引数が無いときの既定値は `n=3`, `x=2.0`).
引数を読み取る処理が抜けているので, 現状では引数を何にしても既定値 (`2.0^3`) のままになる.

`TODO` の箇所に **引数を数値に変換して `n`, `x` に代入する処理** を書け (引数が与えられたときだけ).

- C++: `if (argc > 1) n = atoi(argv[1]);` と `if (argc > 2) x = atof(argv[2]);`
  - `argc` は引数の個数 (プログラム名を含む), `argv[1]`, `argv[2]` が実際の引数 (文字列).
- Fortran: `command_argument_count()` で個数を見て, `get_command_argument` で取り出し, 内部 `read (arg, *) n` で数値化.

## コンパイルと実行

```
# C++
nvc++ -fast args.cpp -o args.exe

# Fortran
nvfortran -fast args.f90 -o args.exe
```

```
./args.exe          # 既定値 2.0 ^ 3
./args.exe 10 1.5    # 1.5 ^ 10
```

## 期待される結果

```
$ ./args.exe 10 1.5
1.500000 ^ 10 = 57.665039
```

引数の読み取りを書く前は, 引数を変えても常に `2.0 ^ 3 = 8` のままになる.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ args.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  /* 第1引数を整数 n, 第2引数を実数 x として受け取り, x の n 乗を表示する.
     引数が無いときの既定値は n=3, x=2.0 */
  int n = 3;
  double x = 2.0;
  // TODO: argv[1] を atoi で n に, argv[2] を atof で x に変換せよ (引数があるときだけ).
  double p = 1.0;
  for (int i = 0; i < n; i++) p *= x;
  printf("%f ^ %d = %f\n", x, n, p);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore args.cpp -o args_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./args_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ args.f90
program args
  character(len=64) :: arg
  integer :: n, i
  real(8) :: x, p
  ! 第1引数を整数 n, 第2引数を実数 x として受け取り, x の n 乗を表示する.
  ! 引数が無いときの既定値は n=3, x=2.0
  n = 3
  x = 2.0d0
  ! TODO: 1番目の引数を n に, 2番目の引数を x に, 内部 read で変換せよ (引数があるときだけ).
  p = 1.0d0
  do i = 1, n
     p = p * x
  end do
  print "(f0.6,a,i0,a,f0.6)", x, " ^ ", n, " = ", p
end program args

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore args.f90 -o args_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./args_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:args.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:args.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:args.cpp}